# Epic Daily ETL Progress Dashboard
Server: rhnv-edwprod.rhnv.hosted  
Database: EDWAdmin

In [1]:
# Install required packages if needed
# !pip install pyodbc pandas plotly

In [2]:
import pyodbc
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from datetime import datetime

In [3]:
# Database connection parameters
server = 'rhnv-edwprod.rhnv.hosted'
database = 'EDWAdmin'

# Connection string using Windows Authentication
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'

# SQL Query
query = """
select th.StatusCD, count(*)
from EDWAdmin.CatalystAdmin.ETLBatchHistoryBASE bh
join EDWAdmin.CatalystAdmin.ETLTableHistoryBASE th on bh.BatchID = th.BatchID
where 1=1
and bh.BatchID in (339390,339389)
group by th.StatusCD
"""

# Execute query and load results
try:
    conn = pyodbc.connect(conn_str)
    df = pd.read_sql(query, conn)
    df.columns = ['StatusCD', 'Count']
    conn.close()
    print(f"Query executed successfully at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\nTotal records: {df['Count'].sum()}")
    display(df)
except Exception as e:
    print(f"Error: {e}")

C:\Users\58811.WMCDOMAIN\AppData\Local\Temp\ipykernel_6876\1788162686.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Query executed successfully at 2026-02-11 15:06:45

Total records: 536


,StatusCD,Count
0,Succeeded,536


In [4]:
# Generate Epic Daily ETL Progress Dashboard

# Define status colors
status_colors = {
    'C': '#28a745',  # Completed - Green
    'F': '#dc3545',  # Failed - Red
    'R': '#007bff',  # Running - Blue
    'P': '#ffc107',  # Pending - Yellow
    'W': '#6c757d'   # Waiting - Gray
}

# Map colors to data
colors = [status_colors.get(status, '#17a2b8') for status in df['StatusCD']]

# Create subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Status Distribution', 'Status Breakdown', 'Progress Summary', ''),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'indicator', 'colspan': 2}, None]],
    row_heights=[0.6, 0.4]
)

# Pie chart
fig.add_trace(
    go.Pie(
        labels=df['StatusCD'],
        values=df['Count'],
        marker=dict(colors=colors),
        textinfo='label+percent+value',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    ),
    row=1, col=1
)

# Bar chart
fig.add_trace(
    go.Bar(
        x=df['StatusCD'],
        y=df['Count'],
        marker=dict(color=colors),
        text=df['Count'],
        textposition='auto',
        hovertemplate='<b>Status: %{x}</b><br>Count: %{y}<extra></extra>'
    ),
    row=1, col=2
)

# Total indicator
total_count = df['Count'].sum()
fig.add_trace(
    go.Indicator(
        mode='number',
        value=total_count,
        title={'text': 'Total Tables Processed'},
        number={'font': {'size': 60}}
    ),
    row=2, col=1
)

# Update layout
fig.update_layout(
    title={
        'text': f'<b>Epic Daily ETL Progress</b><br><sub>Batches: 339390, 339389 | Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 24}
    },
    showlegend=False,
    height=800,
    template='plotly_white'
)

fig.update_xaxes(title_text='Status Code', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=2)

fig.show()

# Print summary statistics
print("\n" + "="*50)
print("EPIC DAILY ETL PROGRESS SUMMARY")
print("="*50)
print(f"Batch IDs: 339390, 339389")
print(f"Total Tables: {total_count}")
print("\nStatus Breakdown:")
for idx, row in df.iterrows():
    percentage = (row['Count'] / total_count) * 100
    print(f"  {row['StatusCD']}: {row['Count']:>5} ({percentage:>5.1f}%)")
print("="*50)


EPIC DAILY ETL PROGRESS SUMMARY
Batch IDs: 339390, 339389
Total Tables: 536

Status Breakdown:
  Succeeded:   536 (100.0%)
